<a href="https://colab.research.google.com/github/zFonta/CEIA-TF-Chess-DL/blob/main/notebooks/06_train_transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 06 - Transformer

Tarea **4.8** del WBS: segunda arquitectura, entrenada con el mismo presupuesto
que la ResNet y comparada contra ella.

**Por qué un transformer.** La ResNet lee el tablero como si fuera una imagen:
a través de ventanas de 3×3, de modo que una relación entre dos casillas lejanas
solo existe después de que suficientes capas se apilaron para cubrir la
distancia. Un alfil en c1 apuntando a h6 está a cinco casillas: varios bloques de
indirección.

La atención elimina esa distancia. El tablero se corta en **64 tokens, uno por
casilla**, y cada capa deja que cualquier casilla mire a cualquier otra en un
solo paso. Clavadas, baterías, una pieza colgada del otro lado del tablero: un
salto de atención en vez de cinco convoluciones.

Si eso alcanza para ganarle a la ResNet es exactamente lo que esta notebook mide.

**La comparación es al mismo presupuesto**, que es la única forma de que la
respuesta signifique algo:

| | ResNet | Transformer |
|---|---|---|
| Configuración | 128 canales × 8 bloques | `d_model` 192 × 6 capas × 8 cabezas |
| Parámetros | 2.913.345 | 2.735.361 |
| Entrada | 18 planos de 8×8 | **los mismos** 18 planos |
| Partición, caché, pérdida, métricas | — | **las mismas** |

Si el transformer ganara con el triple de parámetros no se sabría si ganó la
atención o el tamaño.

**Referencias a batir** (sobre validación, que es contra lo que se decide):

| | val RMSE |
|---|---|
| Piso de material (ajuste lineal) | 0,3973 |
| ResNet campaña 1 | 0,2584 |
| **ResNet campaña 2 (warm-up, 30 épocas)** | **0,2494** |

Sobre test la campaña 2 cerró en **0,2511** (R² 0,736, signo 87,80 %, ρ 0,8385).
El test se toca una sola vez, al final.

## 1. Entorno

In [ ]:
# Clonar el repositorio e instalar el paquete.
# Idempotente: se puede volver a correr tal cual despues de una desconexion.
import os, sys, subprocess, importlib
from pathlib import Path

REPO_DIR = Path("/content/CEIA-TF-Chess-DL")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "https://github.com/zFonta/CEIA-TF-Chess-DL.git", str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)

os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[dev]"], check=True)

# pip registra el paquete editable con un .pth, y los .pth solo se procesan al
# arrancar el interprete: un kernel que ya esta corriendo no lo ve. Agregar src/
# a sys.path lo hace visible sin tener que reiniciar el runtime.
src = str(REPO_DIR / "src")
if src not in sys.path:
    sys.path.insert(0, src)
importlib.invalidate_caches()

# Stockfish con version fija. Entrenar no lo usa, pero sin el se saltean los 17
# tests de integracion del pipeline, que son la evidencia del requerimiento 3.2.
subprocess.run(["bash", "scripts/setup_stockfish.sh"], check=True)
os.environ["PATH"] = f"{REPO_DIR}/bin:" + os.environ["PATH"]

print("Listo. Directorio de trabajo:", os.getcwd())

In [ ]:
import numpy as np
import torch
from chessdl.colab import TRAINING, describe_runtime

runtime = describe_runtime(phase=TRAINING)
print("GPU  :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "sin GPU")
for aviso in runtime.warnings():
    print("AVISO:", aviso)

## 2. Tests

In [ ]:
!{sys.executable} -m pytest -q

## 3. Dataset, partición, caché y pisos

Idéntico a las notebooks 04 y 05 — **a propósito**: la comparación entre las dos
arquitecturas solo vale si ven exactamente las mismas partidas en el mismo
reparto. Si el caché ya está en disco no se reconstruye.

In [ ]:
from chessdl.config import load_config
from chessdl import hf
from chessdl.data import schema
from chessdl.training.split import describe_split, leaked_games, split_masks
from chessdl.training.cache import build_cache, cache_path_for, load_cache
from chessdl.training.baselines import material_baseline, mean_baseline

cfg = load_config()
token = hf.get_token()
directorio = hf.download_dataset(cfg.output.hf_repo_id, "/content/ceia-chess/hub")
tabla = schema.read_dataset(schema.shard_paths(directorio))

fens     = tabla["fen"].to_pylist()
game_ids = tabla["game_id"].to_pylist()
targets  = np.asarray(tabla["value_stm"], dtype=np.float32)

split_cfg = cfg.training.split_config()
masks = split_masks(game_ids, split_cfg)
assert not leaked_games(masks, game_ids)
print(describe_split(masks, game_ids))

ruta_cache = cache_path_for(cfg.training.cache_dir)
if not ruta_cache.exists():
    build_cache(fens, ruta_cache, progress=True)
cache = load_cache(ruta_cache, expected_rows=len(fens))

idx_train = np.flatnonzero(masks['train'])
idx_val   = np.flatnonzero(masks['val'])
idx_test  = np.flatnonzero(masks['test'])

rng = np.random.default_rng(0)
sub = np.sort(rng.choice(idx_train, size=min(200_000, len(idx_train)), replace=False))
media = mean_baseline(targets[idx_train], targets[idx_val])
material, _ = material_baseline(np.asarray(cache[sub]), targets[sub],
                                np.asarray(cache[idx_val]), targets[idx_val])
pisos = {'media': media.rmse, 'material': material.rmse}
print(media); print(material)

## 4. La arquitectura

Tres decisiones de diseño, cada una de las cuales podría razonablemente haber
ido para el otro lado:

**Embeddings posicionales aprendidos, no sinusoidales.** Un tablero no es una
secuencia: la "distancia" entre a1 y a2 no es comparable con la distancia entre
a1 y b1 de ninguna forma que capture una sinusoide. Hay 64 posiciones y no
cambian nunca, así que cada una recibe su propio vector aprendido y la red
deduce la geometría de los datos.

**Pre-norm (`norm_first=True`).** Los transformers post-norm son notoriamente
difíciles de arrancar desde cero sin un calentamiento largo, y acá el
presupuesto es una sesión de Colab, no un cluster. Pre-norm deja el camino del
gradiente limpio desde el primer paso.

**Los mismos 18 planos que la ResNet.** El costo es que los cinco planos
globales —derechos de enroque y reloj de 50 jugadas— son constantes en todas las
casillas y por lo tanto se repiten en los 64 tokens. La proyección de entrada es
libre de colapsar esa redundancia, y pagarla sale más barato que mantener una
segunda codificación y arriesgar que la comparación mida dos cosas a la vez.

El *pooling* (`cls` contra `mean`) es una pregunta empírica y quedó como campo
de configuración; se arranca con `cls`.

In [ ]:
from chessdl.models.transformer import ChessTransformer, TransformerConfig
from chessdl.models.resnet import ChessResNet, ResNetConfig
from chessdl.training.loop import seed_everything

arquitectura = TransformerConfig(d_model=192, layers=6, heads=8, feedforward=768,
                                 dropout=0.1, pooling='cls')

# La siembra va ANTES de construir el modelo: train() tambien siembra, pero para
# entonces los pesos iniciales ya salieron del estado en que estuviera el
# interprete, y no serian reproducibles entre sesiones.
seed_everything(cfg.training.split_seed)
modelo = ChessTransformer(arquitectura)

resnet = ChessResNet(ResNetConfig(channels=128, blocks=8)).count_parameters()
print(modelo.describe())
print(f'ResNet de referencia: {resnet:,} parametros '
      f'({modelo.count_parameters() / resnet - 1:+.1%})')

### Control de cordura antes de gastar GPU

Una arquitectura que *corre* pero no aprende se ve igual que una que sí, y la
diferencia aparece una hora después. Dos chequeos de segundos: que la salida
respete el rango del requerimiento 1.4 por construcción, y que la predicción
dependa de dónde están las piezas —sin señal posicional útil la atención es
invariante a permutaciones y el modelo vería una *bolsa* de piezas sin
geometría.

In [ ]:
import chess
from chessdl.encoding import board_to_tensor
from chessdl.training.cache import cached_to_tensor

muestra = torch.from_numpy(cached_to_tensor(np.asarray(cache[idx_val[:256]])))
with torch.no_grad():
    salida = modelo.eval()(muestra)

print(f'rango de salida  : [{salida.min():.4f}, {salida.max():.4f}]  (req. 1.4: [-1, 1])')
print(f'desvio            : {salida.std():.4f}  (si fuera ~0, la cabeza ignora el tablero)')
assert salida.min() >= -1 and salida.max() <= 1
assert salida.std() > 1e-4


def evaluar(casilla):
    tablero = chess.Board()
    tablero.clear()
    tablero.set_piece_at(chess.E1, chess.Piece(chess.KING, chess.WHITE))
    tablero.set_piece_at(chess.E8, chess.Piece(chess.KING, chess.BLACK))
    tablero.set_piece_at(casilla, chess.Piece(chess.QUEEN, chess.WHITE))
    with torch.no_grad():
        return float(modelo(torch.from_numpy(board_to_tensor(tablero)).unsqueeze(0)))

# Misma dama, distinta casilla. Sin señal posicional estos dos numeros serian identicos.
print(f'dama en d4        : {evaluar(chess.D4):+.4f}')
print(f'dama en h1        : {evaluar(chess.H1):+.4f}')

## 5. Hiperparámetros

No se heredan tal cual los de la ResNet, y conviene decir por qué: **un
transformer no tolera el mismo learning rate que una red convolucional con
BatchNorm.**

| | ResNet (campaña 2) | Transformer | Por qué |
|---|---|---|---|
| `learning_rate` | 1e-3 | **3e-4** | Sin BatchNorm que reescale las activaciones, 1e-3 suele divergir o estancarse alto en las primeras épocas |
| `weight_decay` | 1e-4 | **1e-2** | El valor habitual de AdamW en transformers; casi todos los parámetros están en capas densas |
| `warmup_epochs` | 2 (lo ganó el barrido) | **3** | En la ResNet el calentamiento fue una mejora; acá es prácticamente un requisito de la arquitectura |
| `batch_size`, pérdida, épocas, semilla | — | **idénticos** | Es lo que mantiene comparable la comparación |

Esto es una **primera campaña**, no una configuración optimizada: el equivalente
del barrido de la tarea 4.5 para esta arquitectura queda pendiente, y `run_sweep`
ya acepta `model_factory` para correrlo sin duplicar el código.

> **Es reanudable.** Después de cada época se empuja el estado completo al Hub.
> Si Colab corta, se vuelve a correr la celda de entrenamiento y sigue desde la
> última época: se pierde una época, no la campaña.

In [ ]:
from chessdl.training.checkpoint import HubCheckpoints
from chessdl.training.experiments import transformer_summary
from chessdl.training.loop import train

EPOCAS = 30

repo_modelos = f'{cfg.output.hf_namespace}/{cfg.training.hf_models_repo}'
checkpoints = HubCheckpoints(
    repo_id=repo_modelos,
    run_name='transformer-campana1',
    local_dir='/content/ceia-chess/checkpoints',
    token=token,
    enabled=cfg.training.push_to_hub and token is not None,
)

campana = train(
    modelo, cache, targets, idx_train, idx_val,
    epochs=EPOCAS,
    batch_size=cfg.training.batch_size,
    learning_rate=3e-4,
    weight_decay=1e-2,
    loss_name=cfg.training.loss,
    warmup_epochs=3,
    seed=cfg.training.split_seed,
    checkpoints=checkpoints,
    model_config=transformer_summary(arquitectura),
    baselines=pisos,
)
print()
print(campana.summary())

## 6. Curvas

In [ ]:
import matplotlib.pyplot as plt
from chessdl import viz

viz.apply_style()

RESNET_VAL = 0.2494   # mejor validacion de la campana 2 (notebook 05)

epocas = [e.epoch for e in campana.history.epochs]
fig, ax = plt.subplots()
ax.plot(epocas, [e.val_rmse for e in campana.history.epochs],
        marker='o', markersize=3, color=viz.SERIES[0], label='transformer')
ax.axhline(RESNET_VAL, color=viz.INK_SECONDARY, linestyle='--', linewidth=1.5)
ax.text(1, RESNET_VAL, '  ResNet campana 2', color=viz.INK_SECONDARY, fontsize=8, va='bottom')
ax.axhline(pisos['material'], color=viz.INK_MUTED, linestyle=':', linewidth=1)
ax.text(1, pisos['material'], '  piso de material', color=viz.INK_MUTED, fontsize=8, va='bottom')
ax.legend(loc='upper right')
viz.label_axes(ax, 'Transformer contra la ResNet', 'epoca', 'RMSE sobre validacion',
               note='Mismo presupuesto de parametros, misma particion, mismas etiquetas')
fig.tight_layout()

In [ ]:
# Sobreajuste y calentamiento, los dos diagnosticos que decidieron la campana 2.
fig, (izq, der) = plt.subplots(1, 2, figsize=(11, 4))

izq.plot(epocas, [e.val_rmse ** 2 / e.train_loss for e in campana.history.epochs],
         marker='o', markersize=3, color=viz.SERIES[0])
izq.axhline(1.0, color=viz.INK_MUTED, linestyle=':', linewidth=1)
viz.label_axes(izq, 'Cuanto se separa la validacion', 'epoca',
               'error de validacion / perdida de entrenamiento',
               note='La ResNet llego a 3,75 en la epoca 30')

der.plot(epocas, [e.learning_rate for e in campana.history.epochs],
         marker='o', markersize=3, color=viz.SERIES[1])
viz.label_axes(der, 'Learning rate efectivo', 'epoca', 'lr',
               note='3 epocas de calentamiento lineal y despues coseno')
fig.tight_layout()

## 7. Evaluación final sobre test

El split de test se toca **una sola vez**, con el mejor checkpoint por
validación de esta campaña.

In [ ]:
from chessdl.training.checkpoint import BEST_NAME, load_checkpoint
from chessdl.training.loop import evaluate_split

dispositivo = 'cuda' if torch.cuda.is_available() else 'cpu'
mejor = checkpoints.fetch(BEST_NAME)
if mejor is not None:
    load_checkpoint(mejor, modelo, map_location=dispositivo)
modelo = modelo.to(dispositivo)

test_tr = evaluate_split(modelo, cache, targets, idx_test, dispositivo)
print('TEST -- transformer')
print(test_tr.summary())

## 8. Resumen: atención contra convolución

In [ ]:
TEST_RESNET_C1 = 0.2609   # notebook 04
TEST_RESNET_C2 = 0.2511   # notebook 05

print(f"{'':<34}{'RMSE':>10}{'R2':>10}")
print('-' * 56)
for nombre, valor in [('media constante', pisos['media']),
                      ('material lineal', pisos['material']),
                      ('ResNet campana 1 (test)', TEST_RESNET_C1),
                      ('ResNet campana 2 (test)', TEST_RESNET_C2),
                      ('Transformer campana 1 (test)', test_tr.rmse)]:
    print(f"{nombre:<34}{valor:>10.4f}{1-(valor/pisos['media'])**2:>10.3f}")

delta = (TEST_RESNET_C2 - test_tr.rmse) / TEST_RESNET_C2
print()
print(f"Contra la mejor ResNet: el transformer {'mejora' if delta > 0 else 'EMPEORA'} {abs(delta):.2%}.")
print()
if delta > 0.02:
    print('Gana la atencion, y con margen. El siguiente paso es el barrido de')
    print('hiperparametros de esta arquitectura (run_sweep con model_factory).')
elif delta > -0.02:
    print('Empate tecnico. Con presupuestos equiparados las dos arquitecturas')
    print('llegan al mismo lugar, lo que apunta a que el limite no esta en como')
    print('se lee el tablero sino en los datos: se consumio el 47 % del extracto.')
else:
    print('Gana la ResNet. Vale la pena descartar antes que sea falta de ajuste:')
    print('esta es la primera configuracion probada del transformer, contra una')
    print('ResNet que ya paso por un barrido de cinco brazos.')
print()
print(f'Pesos y metricas: https://huggingface.co/{repo_modelos}')